In [1]:
"""
PyTorch Tutorial for Beginners
Linear Regression and Classification Models with Loss Functions and Optimizers
"""

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# ============================================================================
# PART 1: LINEAR REGRESSION MODEL
# ============================================================================
print("\n" + "="*80)
print("PART 1: LINEAR REGRESSION")
print("="*80)

# 1.1 Generate synthetic data for linear regression
print("\n1.1 Generating synthetic data...")
X_train = np.linspace(0, 10, 100).reshape(-1, 1)
y_train = 3 * X_train + 7 + np.random.randn(100, 1) * 2

# Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train)

print(f"Training data shape: X={X_train_tensor.shape}, y={y_train_tensor.shape}")

# 1.2 Create a custom Dataset
class RegressionDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# 1.3 Create DataLoader
dataset = RegressionDataset(X_train_tensor, y_train_tensor)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)
print(f"DataLoader created with batch_size=16, total batches={len(dataloader)}")

# 1.4 Define Linear Regression Model
class LinearRegressionModel(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(LinearRegressionModel, self).__init__()
        self.linear = nn.Linear(input_dim, output_dim)
    
    def forward(self, x):
        return self.linear(x)

# Initialize model
linear_model = LinearRegressionModel(input_dim=1, output_dim=1)
print(f"\n1.4 Model architecture:\n{linear_model}")

# Print initial parameters
print("\nInitial parameters:")
for name, param in linear_model.named_parameters():
    print(f"{name}: {param.data}")

# 1.5 Define Loss Function
criterion_mse = nn.MSELoss()  # Mean Squared Error
criterion_l1 = nn.L1Loss()     # Mean Absolute Error (alternative)
print(f"\n1.5 Loss function: MSE (Mean Squared Error)")

# 1.6 Define Optimizer
optimizer_sgd = optim.SGD(linear_model.parameters(), lr=0.01)
# Alternative optimizers:
# optimizer_adam = optim.Adam(linear_model.parameters(), lr=0.01)
# optimizer_rmsprop = optim.RMSprop(linear_model.parameters(), lr=0.01)
print(f"1.6 Optimizer: SGD with learning rate=0.01")

# 1.7 Training Loop
print("\n1.7 Training Linear Regression Model...")
num_epochs = 100
losses = []

for epoch in range(num_epochs):
    epoch_loss = 0
    
    for batch_X, batch_y in dataloader:
        # Forward pass
        predictions = linear_model(batch_X)
        loss = criterion_mse(predictions, batch_y)
        
        # Backward pass
        optimizer_sgd.zero_grad()  # Clear gradients
        loss.backward()            # Compute gradients
        optimizer_sgd.step()       # Update parameters
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(dataloader)
    losses.append(avg_loss)
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

# Print final parameters
print("\nFinal parameters:")
for name, param in linear_model.named_parameters():
    print(f"{name}: {param.data}")

# 1.8 Model Evaluation
linear_model.eval()  # Set to evaluation mode
with torch.no_grad():  # Disable gradient computation
    predictions = linear_model(X_train_tensor)
    final_loss = criterion_mse(predictions, y_train_tensor)
    print(f"\n1.8 Final MSE Loss: {final_loss.item():.4f}")

# ============================================================================
# PART 2: BINARY CLASSIFICATION MODEL
# ============================================================================
print("\n" + "="*80)
print("PART 2: BINARY CLASSIFICATION")
print("="*80)

# 2.1 Generate synthetic data for binary classification
print("\n2.1 Generating synthetic classification data...")
from sklearn.datasets import make_classification

X_class, y_class = make_classification(
    n_samples=500, n_features=2, n_redundant=0, 
    n_informative=2, n_clusters_per_class=1, 
    random_state=42
)

# Convert to PyTorch tensors
X_class_tensor = torch.FloatTensor(X_class)
y_class_tensor = torch.FloatTensor(y_class).view(-1, 1)

print(f"Classification data shape: X={X_class_tensor.shape}, y={y_class_tensor.shape}")

# 2.2 Create DataLoader for classification
class_dataset = TensorDataset(X_class_tensor, y_class_tensor)
class_dataloader = DataLoader(class_dataset, batch_size=32, shuffle=True)
print(f"DataLoader created with batch_size=32, total batches={len(class_dataloader)}")

# 2.3 Define Binary Classification Model
class BinaryClassifier(nn.Module):
    def __init__(self, input_dim):
        super(BinaryClassifier, self).__init__()
        self.layer1 = nn.Linear(input_dim, 16)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.layer2 = nn.Linear(16, 8)
        self.output = nn.Linear(8, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.layer2(x)
        x = self.relu(x)
        x = self.output(x)
        x = self.sigmoid(x)
        return x

# Initialize classifier
classifier = BinaryClassifier(input_dim=2)
print(f"\n2.3 Model architecture:\n{classifier}")

# Count parameters
total_params = sum(p.numel() for p in classifier.parameters())
trainable_params = sum(p.numel() for p in classifier.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params}")
print(f"Trainable parameters: {trainable_params}")

# 2.4 Define Loss Function for Classification
criterion_bce = nn.BCELoss()  # Binary Cross Entropy
# Alternative: BCEWithLogitsLoss (combines sigmoid + BCE for numerical stability)
# criterion_bce_logits = nn.BCEWithLogitsLoss()
print(f"\n2.4 Loss function: Binary Cross Entropy (BCE)")

# 2.5 Define Optimizer (using Adam this time)
optimizer_adam = optim.Adam(classifier.parameters(), lr=0.001)
# Can also use a learning rate scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer_adam, step_size=50, gamma=0.5)
print(f"2.5 Optimizer: Adam with learning rate=0.001")
print(f"    Scheduler: StepLR (decay by 0.5 every 50 epochs)")

# 2.6 Training Loop for Classifier
print("\n2.6 Training Binary Classifier...")
num_epochs_class = 150
class_losses = []
accuracies = []

for epoch in range(num_epochs_class):
    classifier.train()  # Set to training mode
    epoch_loss = 0
    correct = 0
    total = 0
    
    for batch_X, batch_y in class_dataloader:
        # Forward pass
        predictions = classifier(batch_X)
        loss = criterion_bce(predictions, batch_y)
        
        # Backward pass
        optimizer_adam.zero_grad()
        loss.backward()
        optimizer_adam.step()
        
        epoch_loss += loss.item()
        
        # Calculate accuracy
        predicted_classes = (predictions > 0.5).float()
        correct += (predicted_classes == batch_y).sum().item()
        total += batch_y.size(0)
    
    # Update learning rate
    scheduler.step()
    
    avg_loss = epoch_loss / len(class_dataloader)
    accuracy = 100 * correct / total
    class_losses.append(avg_loss)
    accuracies.append(accuracy)
    
    if (epoch + 1) % 30 == 0:
        current_lr = optimizer_adam.param_groups[0]['lr']
        print(f"Epoch [{epoch+1}/{num_epochs_class}], Loss: {avg_loss:.4f}, "
              f"Accuracy: {accuracy:.2f}%, LR: {current_lr:.6f}")

# 2.7 Model Evaluation
print("\n2.7 Evaluating Binary Classifier...")
classifier.eval()
with torch.no_grad():
    predictions = classifier(X_class_tensor)
    predicted_classes = (predictions > 0.5).float()
    accuracy = (predicted_classes == y_class_tensor).sum().item() / len(y_class_tensor)
    final_loss = criterion_bce(predictions, y_class_tensor)
    
    print(f"Final BCE Loss: {final_loss.item():.4f}")
    print(f"Final Accuracy: {accuracy * 100:.2f}%")

# ============================================================================
# PART 3: MULTI-CLASS CLASSIFICATION
# ============================================================================
print("\n" + "="*80)
print("PART 3: MULTI-CLASS CLASSIFICATION")
print("="*80)

# 3.1 Generate synthetic data for multi-class classification
print("\n3.1 Generating multi-class classification data...")
X_multi, y_multi = make_classification(
    n_samples=600, n_features=4, n_informative=3, 
    n_redundant=0, n_classes=3, n_clusters_per_class=1, 
    random_state=42
)

# Convert to PyTorch tensors
X_multi_tensor = torch.FloatTensor(X_multi)
y_multi_tensor = torch.LongTensor(y_multi)  # Use LongTensor for class indices

print(f"Multi-class data shape: X={X_multi_tensor.shape}, y={y_multi_tensor.shape}")
print(f"Classes: {torch.unique(y_multi_tensor).tolist()}")

# 3.2 Create DataLoader
multi_dataset = TensorDataset(X_multi_tensor, y_multi_tensor)
multi_dataloader = DataLoader(multi_dataset, batch_size=32, shuffle=True)

# 3.3 Define Multi-Class Classification Model
class MultiClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(MultiClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Dropout(0.3),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, num_classes)
        )
    
    def forward(self, x):
        return self.network(x)

# Initialize model
multi_classifier = MultiClassifier(input_dim=4, num_classes=3)
print(f"\n3.3 Model architecture:\n{multi_classifier}")

# 3.4 Define Loss Function (CrossEntropyLoss)
criterion_ce = nn.CrossEntropyLoss()  # Combines LogSoftmax + NLLLoss
print(f"\n3.4 Loss function: Cross Entropy Loss")

# 3.5 Define Optimizer
optimizer_adamw = optim.AdamW(multi_classifier.parameters(), lr=0.001, weight_decay=0.01)
print(f"3.5 Optimizer: AdamW with learning rate=0.001, weight_decay=0.01")

# 3.6 Training Loop
print("\n3.6 Training Multi-Class Classifier...")
num_epochs_multi = 100
multi_losses = []
multi_accuracies = []

for epoch in range(num_epochs_multi):
    multi_classifier.train()
    epoch_loss = 0
    correct = 0
    total = 0
    
    for batch_X, batch_y in multi_dataloader:
        # Forward pass
        outputs = multi_classifier(batch_X)
        loss = criterion_ce(outputs, batch_y)
        
        # Backward pass
        optimizer_adamw.zero_grad()
        loss.backward()
        
        # Gradient clipping (optional but good practice)
        torch.nn.utils.clip_grad_norm_(multi_classifier.parameters(), max_norm=1.0)
        
        optimizer_adamw.step()
        
        epoch_loss += loss.item()
        
        # Calculate accuracy
        _, predicted = torch.max(outputs.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
    
    avg_loss = epoch_loss / len(multi_dataloader)
    accuracy = 100 * correct / total
    multi_losses.append(avg_loss)
    multi_accuracies.append(accuracy)
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs_multi}], Loss: {avg_loss:.4f}, "
              f"Accuracy: {accuracy:.2f}%")

# 3.7 Evaluation
print("\n3.7 Evaluating Multi-Class Classifier...")
multi_classifier.eval()
with torch.no_grad():
    outputs = multi_classifier(X_multi_tensor)
    _, predicted = torch.max(outputs, 1)
    accuracy = (predicted == y_multi_tensor).sum().item() / len(y_multi_tensor)
    final_loss = criterion_ce(outputs, y_multi_tensor)
    
    print(f"Final Cross Entropy Loss: {final_loss.item():.4f}")
    print(f"Final Accuracy: {accuracy * 100:.2f}%")

# ============================================================================
# PART 4: SAVING AND LOADING MODELS
# ============================================================================
print("\n" + "="*80)
print("PART 4: SAVING AND LOADING MODELS")
print("="*80)

# 4.1 Save model
print("\n4.1 Saving models...")
torch.save(linear_model.state_dict(), 'linear_model.pth')
torch.save(classifier.state_dict(), 'binary_classifier.pth')
torch.save({
    'epoch': num_epochs_multi,
    'model_state_dict': multi_classifier.state_dict(),
    'optimizer_state_dict': optimizer_adamw.state_dict(),
    'loss': final_loss.item(),
}, 'multi_classifier_checkpoint.pth')
print("Models saved successfully!")

# 4.2 Load model
print("\n4.2 Loading models...")
loaded_linear_model = LinearRegressionModel(1, 1)
loaded_linear_model.load_state_dict(torch.load('linear_model.pth'))
loaded_linear_model.eval()
print("Linear model loaded successfully!")

# Load checkpoint
checkpoint = torch.load('multi_classifier_checkpoint.pth')
loaded_multi_classifier = MultiClassifier(4, 3)
loaded_multi_classifier.load_state_dict(checkpoint['model_state_dict'])
loaded_multi_classifier.eval()
print(f"Multi-class classifier loaded from epoch {checkpoint['epoch']}")

print("\n" + "="*80)
print("TUTORIAL COMPLETE!")
print("="*80)
print("\nKey PyTorch Functions Demonstrated:")
print("- torch.FloatTensor, torch.LongTensor: Tensor creation")
print("- nn.Module: Base class for neural networks")
print("- nn.Linear: Fully connected layer")
print("- nn.ReLU, nn.Sigmoid: Activation functions")
print("- nn.Dropout, nn.BatchNorm1d: Regularization")
print("- nn.MSELoss, nn.BCELoss, nn.CrossEntropyLoss: Loss functions")
print("- optim.SGD, optim.Adam, optim.AdamW: Optimizers")
print("- optim.lr_scheduler.StepLR: Learning rate scheduler")
print("- DataLoader, Dataset, TensorDataset: Data handling")
print("- model.train(), model.eval(): Training/evaluation modes")
print("- torch.no_grad(): Disable gradient computation")
print("- torch.save(), torch.load(): Model persistence")
print("- torch.nn.utils.clip_grad_norm_: Gradient clipping")

PyTorch Version: 2.4.1
CUDA Available: False

PART 1: LINEAR REGRESSION

1.1 Generating synthetic data...
Training data shape: X=torch.Size([100, 1]), y=torch.Size([100, 1])
DataLoader created with batch_size=16, total batches=7

1.4 Model architecture:
LinearRegressionModel(
  (linear): Linear(in_features=1, out_features=1, bias=True)
)

Initial parameters:
linear.weight: tensor([[0.7645]])
linear.bias: tensor([0.8300])

1.5 Loss function: MSE (Mean Squared Error)
1.6 Optimizer: SGD with learning rate=0.01

1.7 Training Linear Regression Model...
Epoch [20/100], Loss: 4.6733
Epoch [40/100], Loss: 3.6150
Epoch [60/100], Loss: 3.4258
Epoch [80/100], Loss: 3.2715
Epoch [100/100], Loss: 3.2835

Final parameters:
linear.weight: tensor([[3.1283]])
linear.bias: tensor([6.5318])

1.8 Final MSE Loss: 3.4910

PART 2: BINARY CLASSIFICATION

2.1 Generating synthetic classification data...
Classification data shape: X=torch.Size([500, 2]), y=torch.Size([500, 1])
DataLoader created with batch_size=

/var/folders/2p/86zsbczs7dl_1s8zyqmz_j8r0000gn/T/ipykernel_35050/2274557237.py:380: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  loaded_linear_model.load_state_dict(torch.l